In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(19)

In [2]:
Q = torch.randn(3,4)
K = torch.randn(3,4)
V = torch.randn(3,4)

In [3]:
# Now let's compute self attention in matrix form
def attention(Q, K, V, mask=None):
  # trin 1: Dot product, we want words x words. We have two matrix of words x numbers.
  # the order is important as rows = queries and columns = keys
  # We have to define that its the last 2 dimension to transpose such that we always transpose the words and numbers. This was an error when heads and teksts were introduced.
  dot_product_matrix = Q @ K.transpose(-2,-1)
  print(dot_product_matrix.shape)

  # trin 2: scale by sqrt(dimension), we use Q.shape[-1] to make sure we get number of dimensions pr. word.
  scaled_dot_product = dot_product_matrix / math.sqrt(Q.shape[-1])

  # trin ekstra: mask, teksts have padding so they are the same length. this is important so the tensors have the same shape.
  # we do this, by making empty spaces = -infinty before the softmax, since softmax then makes them 0
  # This makes it so the words never pay attention to the padding. As softmax makes it 0%.
  # So the mask is a table with 1 if there is a word and 0 if this it padding.
  if mask is not None:
    scaled_dot_product = scaled_dot_product.masked_fill(mask == 0, -math.inf)

  # trin 3: softmax af attention, we use dim=-1 to always get the last dimension, which is keys.
  softmax_attention = torch.softmax(scaled_dot_product, dim=-1)
  print(softmax_attention)
  
  # trin 4: Every word becomes a weighted combinations of all the words values
  X_prime = softmax_attention @ V

  return X_prime, softmax_attention

In [4]:
##### TEST IF WORKS,  DELETE


# Run the attention function
X_prime, attention_weights = attention(Q, K, V)

# Check 1: form, skal være torch.Size([3, 4]): 3 ord, 4 tal pr. ord
print("Form af X_prime:", X_prime.shape)

# Check 2: hver række i attention skal summe til 1 (Notebook 12.1: "should be positive sum to one")
print("Rækkesummer:", attention_weights.sum(dim=-1))

# Selve resultatet
print(X_prime)

torch.Size([3, 3])
tensor([[0.2442, 0.5680, 0.1878],
        [0.0136, 0.0105, 0.9759],
        [0.0717, 0.3454, 0.5829]])
Form af X_prime: torch.Size([3, 4])
Rækkesummer: tensor([1., 1., 1.])
tensor([[-0.3819,  0.3114, -0.3864, -1.3563],
        [ 0.3922, -0.5987, -0.4263, -1.1151],
        [-0.1947, -0.0974, -0.4804, -1.2937]])


In [5]:
Q = torch.randn(2,2,3,4)
K = torch.randn(2,2,3,4)
V = torch.randn(2,2,3,4)

In [6]:
attention(Q,K,V)

torch.Size([2, 2, 3, 3])
tensor([[[[0.3645, 0.1752, 0.4604],
          [0.1796, 0.4034, 0.4170],
          [0.8902, 0.0535, 0.0563]],

         [[0.2767, 0.3314, 0.3919],
          [0.2787, 0.4206, 0.3007],
          [0.2565, 0.4820, 0.2615]]],


        [[[0.4718, 0.2482, 0.2800],
          [0.4117, 0.2276, 0.3607],
          [0.3827, 0.1936, 0.4236]],

         [[0.2052, 0.4422, 0.3526],
          [0.2439, 0.2815, 0.4747],
          [0.2695, 0.0625, 0.6679]]]])


(tensor([[[[-1.4224e+00,  4.2180e-02, -4.8640e-01,  1.1309e-01],
           [-1.6774e+00,  2.2424e-01, -3.2877e-01,  3.9410e-01],
           [-1.0032e+00, -1.1650e+00, -1.4773e+00,  7.0548e-01]],
 
          [[ 8.0004e-02, -2.1613e-01,  5.9423e-01,  8.7463e-01],
           [ 3.6740e-02, -2.8868e-01,  4.2074e-01,  9.3025e-01],
           [ 1.4970e-03, -3.5102e-01,  3.2851e-01,  9.7834e-01]]],
 
 
         [[[-2.2210e-01, -6.1666e-01, -3.2826e-01,  2.3611e-01],
           [-1.9524e-01, -6.4997e-01, -3.2511e-01,  2.4856e-01],
           [-1.7023e-01, -6.7830e-01, -3.3453e-01,  2.5311e-01]],
 
          [[-4.1007e-01, -4.5338e-01, -1.0032e+00,  6.8031e-01],
           [-3.5835e-01, -6.2888e-01, -1.2444e+00,  7.9447e-01],
           [-2.6719e-01, -8.4301e-01, -1.5565e+00,  9.0539e-01]]]]),
 tensor([[[[0.3645, 0.1752, 0.4604],
           [0.1796, 0.4034, 0.4170],
           [0.8902, 0.0535, 0.0563]],
 
          [[0.2767, 0.3314, 0.3919],
           [0.2787, 0.4206, 0.3007],
           [0.25